# In RAG-2, we build the vector database in this notebook, based on the chunked text in RAG-1.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install -q faiss-cpu openai langchain langchain_openai tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.4/31.4 MB 44.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.6/75.6 kB 4.4 MB/s eta 0:00:00


In [ ]:
!pip install langchain_community

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 27.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 2.5 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.32.5 which is incompatible.


In [ ]:
import os
from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings
import pickle as pkl
from tqdm import tqdm
openai_key = "" # Put your Open_AI Key here
os.environ["OPENAI_API_KEY"] = openai_key

In [ ]:
with open("/content/drive/MyDrive/AccountingandFinance/chunked_dataset_Nvidia.pkl", "rb") as fp:
  chunked_dataset = pkl.load(fp)

In [ ]:
typedoc = "10-K"
company_name = "Nvidia"
ticker = "NVID"
split_size = 15 ### Change this accordingly
overlap = 1 ### Change this accordingly

In [ ]:
# def load_return_vectorstore(embeddings, dbpath, index_name = "FinancialIndex"):
#     vectorstore = FAISS.load_local(folder_path=dbpath,
#                                         embeddings=embeddings,
#                                         index_name=index_name,
#                                         allow_dangerous_deserialization=True)
#     return vectorstore

In [ ]:
embeddings = OpenAIEmbeddings(model="text-embedding-3-large")

for year, _ in tqdm(chunked_dataset.items()):
  if not os.path.exists(f"./content/{company_name}/{year}_{typedoc}_{split_size}_{overlap}/faiss_db"):
    vectorstore = FAISS.from_texts(chunked_dataset[year], embedding=embeddings)
    vectorstore.save_local(folder_path = f"./content/{company_name}/{year}_{typedoc}_{split_size}_{overlap}/faiss_db", index_name = "FinancialIndex")
#这里显示100%说明跑通了，没有就看看RAG1跑通了没

100%|██████████| 10/10 [00:30<00:00,  3.07s/it]


In [ ]:
years = list(chunked_dataset.keys())
vectorstore = FAISS.load_local(folder_path=f"./content/{company_name}/{years[0]}_{typedoc}_{split_size}_{overlap}/faiss_db",
                                  embeddings=embeddings,
                                  index_name="FinancialIndex",
                                  allow_dangerous_deserialization=True)

for i in range(1,len(years)):
  _vectorstore = FAISS.load_local(folder_path=f"./content/{company_name}/{years[i]}_{typedoc}_{split_size}_{overlap}/faiss_db",
                                  embeddings=embeddings,
                                  index_name="FinancialIndex",
                                  allow_dangerous_deserialization=True)
  vectorstore.merge_from( _vectorstore)

In [ ]:
vectorstore.save_local(folder_path = f"./content/{company_name}/Combined/faiss_db", index_name = "FinancialIndex")

In [ ]:
!cp -r /content/content/Nvidia /content/drive/MyDrive/AccountingandFinance/Vectorstores/